In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, accuracy_score, precision_recall_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
import warnings

warnings.filterwarnings("ignore")

# === CONFIGURACIÓN ===
BASE_INPUT_PATH = r"C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\selected"
BASE_OUTPUT_PATH = r"C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\models"
MODEL_NAME = "MLP"
INTENTO = 1
CATEGORIA = "Standard"
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f"{MODEL_NAME}_{INTENTO:02d}_{CATEGORIA}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# === Hiperparámetros reducidos para evitar sobrecarga ===
param_grid = {
    'hidden_layer_sizes': [(50,), (100,)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate': ['constant', 'adaptive']
}

metricas_totales = []
variantes_X = sorted([f for f in os.listdir(BASE_INPUT_PATH) if f.startswith(f"X_train_{CATEGORIA}")])

for x_file in variantes_X:
    base_name = x_file.replace("X_train_", "").replace(".parquet", "")
    try:
        print(f"\n🔍 Procesando: {base_name}")

        X_train = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"X_train_{base_name}.parquet"))
        X_test = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"X_test_{base_name}.parquet"))
        y_train = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"y_train_{base_name}.parquet")).squeeze()
        y_test = pd.read_parquet(os.path.join(BASE_INPUT_PATH, f"y_test_{base_name}.parquet")).squeeze()
        classes = np.unique(y_train)

        start = time.time()
        model = MLPClassifier(max_iter=300, random_state=42)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        grid = GridSearchCV(model, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
        tiempo = (time.time() - start) / 60

        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)
        y_pred_train = best_model.predict(X_train)

        report_test = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T
        report_train = pd.DataFrame(classification_report(y_train, y_pred_train, output_dict=True)).T
        report_test['set'] = 'test'
        report_train['set'] = 'train'
        full_report = pd.concat([report_train, report_test])
        full_report['tiempo_min'] = tiempo
        full_report.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{base_name}.csv"))

        # PDF
        with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{base_name}.pdf")) as pdf:
            ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
            plt.title("Matriz de Confusión")
            pdf.savefig(); plt.close()

            y_bin = label_binarize(y_test, classes=classes)
            for i, clase in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={auc(fpr, tpr):.2f})")
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title("Curvas ROC"); plt.legend(); pdf.savefig(); plt.close()

            for i, clase in enumerate(classes):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                plt.plot(recall, precision, label=f"Clase {clase}")
            plt.title("Curvas Precision-Recall"); plt.legend(); pdf.savefig(); plt.close()

        metricas_totales.append({
            "modelo": base_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "macro_f1_test": report_test.loc['macro avg', 'f1-score'],
            "macro_f1_train": report_train.loc['macro avg', 'f1-score'],
            "sobreajuste_f1": report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
            "tiempo_min": tiempo
        })

        print(f"✅ {base_name}: F1_macro_test={report_test.loc['macro avg', 'f1-score']:.3f}, Tiempo={tiempo:.2f}min")

    except Exception as e:
        print(f"❌ Error en {base_name}: {e}")

# Guardar resumen global
pd.DataFrame(metricas_totales).to_csv(os.path.join(OUTPUT_PATH, f"resumen_modelos_mlp.csv"), index=False)
print("\n📊 Resultados guardados correctamente.")



🔍 Procesando: Standard_FE_ALL
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Standard_FE_ALL: F1_macro_test=0.412, Tiempo=763.78min

🔍 Procesando: Standard_FE_ANOVA
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Standard_FE_ANOVA: F1_macro_test=0.375, Tiempo=125.54min

🔍 Procesando: Standard_FE_MI
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Standard_FE_MI: F1_macro_test=0.397, Tiempo=434.17min

🔍 Procesando: Standard_FE_PCA30
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Standard_FE_PCA30: F1_macro_test=0.344, Tiempo=23.82min

🔍 Procesando: Standard_FE_PCAopt
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Standard_FE_PCAopt: F1_macro_test=0.422, Tiempo=364.71min

🔍 Procesando: Standard_FE_VAR
Fitting 5 folds for each of 8 candidates, totalling 40 fits
✅ Standard_FE_VAR: F1_macro_test=0.412, Tiempo=753.42min

🔍 Procesando: Standard_ORIGINAL_ALL
Fitting 5 folds for each of 8 candidates, totalling 40 fits
❌ Er